Loading cleaned data into SQLite and executing queries

In [0]:
import pandas as pd
import sqlite3
import os

BASE_DIR="/Volumes/assignment8/assignment8schema/assignment8volume"
DB_PATH = "/tmp/ecommerce.db"

customers_df=pd.read_csv(f"{BASE_DIR}/customers_clean.csv")
products_clean=pd.read_csv(f"{BASE_DIR}/products_clean.csv")
orders_clean=pd.read_csv(f"{BASE_DIR}/orders_clean.csv")
order_items_clean=pd.read_csv(f"{BASE_DIR}/order_items_clean.csv")

In [0]:
if os.path.exists(DB_PATH):
    os.remove(DB_PATH)

conn=sqlite3.connect(DB_PATH)

customers_df.to_sql("customers", conn, index=False, if_exists="replace")
products_clean.to_sql("products", conn, index=False, if_exists="replace")
orders_clean.to_sql("orders", conn, index=False, if_exists="replace")
order_items_clean.to_sql("order_items", conn, index=False, if_exists="replace")

conn.execute("CREATE INDEX IF NOT EXISTS idx_orders_customer ON orders(customer_id);")
conn.execute("CREATE INDEX IF NOT EXISTS idx_items_order ON order_items(order_id);")
conn.execute("CREATE INDEX IF NOT EXISTS idx_items_product ON order_items(product_id);")
conn.commit()

def q(sql, params=None):
    return pd.read_sql_query(sql, conn, params=params)

print("Tables loaded:", conn.execute(
    "SELECT name FROM sqlite_master WHERE type='table';"
).fetchall())

Tables loaded: [('customers',), ('products',), ('orders',), ('order_items',)]


In [0]:
sql_q1="""
SELECT
    p.category,
    ROUND(SUM(oi.quantity * oi.unit_price * (1 - oi.discount_percent / 100.0)), 2) AS total_revenue
FROM order_items oi
JOIN products p ON p.product_id = oi.product_id
GROUP BY p.category
ORDER BY total_revenue DESC;
"""
q(sql_q1)

,category,total_revenue
0,Books,16420841.40
1,Clothing,15741224.55
2,Home,14472507.55
3,Electronics,13664931.01


In [0]:
sql_q2="""
SELECT
    o.customer_id,
    ROUND(SUM(oi.quantity * oi.unit_price * (1 - oi.discount_percent / 100.0)), 2) AS total_order_value
FROM orders o
JOIN order_items oi ON oi.order_id = o.order_id
WHERE o.customer_id IS NOT NULL
GROUP BY o.customer_id
ORDER BY total_order_value DESC
LIMIT 10;
"""
q(sql_q2)

,customer_id,total_order_value
0,100.0,332702.26
1,443.0,317403.61
2,97.0,315980.81
3,424.0,298273.06
4,528.0,277768.75
5,2.0,258832.92
6,169.0,256002.86
7,488.0,243793.31
8,176.0,238462.87
9,546.0,235943.40


In [0]:
sql_q3="""
SELECT
    strftime('%Y-%m', order_date) AS order_month,
    COUNT(*) AS order_count
FROM orders
WHERE order_date >= datetime((SELECT MAX(order_date) FROM orders), '-12 months')
GROUP BY order_month
ORDER BY order_month;
"""
q(sql_q3)

,order_month,order_count
0,2025-07,108
1,2025-08,115
2,2025-09,87
3,2025-10,87
4,2025-11,112
5,2025-12,87
6,2026-01,96
7,2026-02,97
8,2026-03,93
9,2026-04,93


In [0]:
sql_q4="""
SELECT DISTINCT o.customer_id
FROM orders o
WHERE o.customer_id IS NOT NULL
  AND o.customer_id NOT IN (
      SELECT customer_id FROM orders WHERE status = 'DELIVERED' AND customer_id IS NOT NULL
  );
"""
q(sql_q4)

,customer_id
0,17.0
1,23.0
2,34.0
3,62.0
4,83.0
5,170.0
6,180.0
7,183.0
8,201.0
9,209.0


In [0]:
sql_q5="""
SELECT
    p.product_id,
    p.product_name,
    SUM(CASE WHEN oi.quantity > 0 THEN oi.quantity ELSE 0 END) AS total_purchased,
    SUM(CASE WHEN oi.quantity < 0 THEN -oi.quantity ELSE 0 END) AS total_returned
FROM order_items oi
JOIN products p ON p.product_id = oi.product_id
GROUP BY p.product_id, p.product_name
HAVING total_returned > total_purchased;
"""
q(sql_q5)

,product_id,product_name,total_purchased,total_returned


In [0]:
sql_q6="""
SELECT
    p.category,
    SUM(CASE WHEN oi.quantity < 0 THEN -oi.quantity ELSE 0 END) AS returned_items,
    SUM(ABS(oi.quantity)) AS total_items,
    ROUND(
        1.0 * SUM(CASE WHEN oi.quantity < 0 THEN -oi.quantity ELSE 0 END)
        / NULLIF(SUM(ABS(oi.quantity)), 0), 4
    ) AS return_rate
FROM order_items oi
JOIN products p ON p.product_id = oi.product_id
GROUP BY p.category
ORDER BY return_rate DESC;
"""
q(sql_q6)

,category,returned_items,total_items,return_rate
0,Clothing,177,5435,0.0326
1,Books,179,5783,0.0310
2,Electronics,128,4708,0.0272
3,Home,132,4887,0.0270


In [0]:
sql_q7="""
WITH daily AS (
    SELECT
        o.region_code,
        DATE(o.order_date) AS order_date,
        SUM(oi.quantity * oi.unit_price * (1 - oi.discount_percent / 100.0)) AS daily_revenue
    FROM orders o
    JOIN order_items oi ON oi.order_id = o.order_id
    GROUP BY o.region_code, DATE(o.order_date)
)
SELECT
    region_code,
    order_date,
    ROUND(daily_revenue, 2) AS daily_revenue,
    ROUND(SUM(daily_revenue) OVER (
        PARTITION BY region_code ORDER BY order_date
        ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
    ), 2) AS running_total
FROM daily
ORDER BY region_code, order_date;
"""
q(sql_q7)

,region_code,order_date,daily_revenue,running_total
0,CENTRAL,2024-01-02,18190.38,18190.38
1,CENTRAL,2024-01-06,-4148.82,14041.56
2,CENTRAL,2024-01-07,18422.57,32464.12
3,CENTRAL,2024-01-08,21906.36,54370.48
4,CENTRAL,2024-01-09,64794.78,119165.26
...,...,...,...,...
2026,WEST,2026-06-21,31632.05,11890747.74
2027,WEST,2026-06-22,6869.23,11897616.96
2028,WEST,2026-06-24,80269.00,11977885.96
2029,WEST,2026-06-28,36185.29,12014071.25


In [0]:
sql_q8="""
WITH product_revenue AS (
    SELECT
        p.category,
        p.product_name,
        SUM(oi.quantity * oi.unit_price * (1 - oi.discount_percent / 100.0)) AS total_revenue
    FROM order_items oi
    JOIN products p ON p.product_id = oi.product_id
    GROUP BY p.category, p.product_name
)
SELECT
    category,
    product_name,
    ROUND(total_revenue, 2) AS total_revenue,
    DENSE_RANK() OVER (PARTITION BY category ORDER BY total_revenue DESC) AS rank_in_category
FROM product_revenue
ORDER BY category, rank_in_category;
"""
q(sql_q8)

,category,product_name,total_revenue,rank_in_category
0,Books,Wide Comic Believe,766931.29,1
1,Books,Better Comic Inside,749839.98,2
2,Books,Large Center,710486.55,3
3,Books,Word Summer,664307.66,4
4,Books,Magazine Watch,661565.76,5
...,...,...,...,...
145,Home,Market With,101600.45,30
146,Home,Score Authority,71654.95,31
147,Home,Tend Guess,52069.88,32
148,Home,Account Participant,50177.91,33


In [0]:
sql_q9="""
WITH cust_orders AS (
    SELECT
        customer_id,
        order_date,
        LAG(order_date) OVER (PARTITION BY customer_id ORDER BY order_date) AS previous_order_date
    FROM orders
    WHERE customer_id IS NOT NULL
),
with_gap AS (
    SELECT
        customer_id,
        order_date,
        previous_order_date,
        CASE WHEN previous_order_date IS NOT NULL
             THEN CAST(julianday(order_date) - julianday(previous_order_date) AS REAL)
             ELSE NULL
        END AS days_gap
    FROM cust_orders
),
avg_gap AS (
    SELECT customer_id, AVG(days_gap) AS avg_gap
    FROM with_gap
    WHERE days_gap IS NOT NULL
    GROUP BY customer_id
)
SELECT
    w.customer_id,
    w.order_date,
    w.previous_order_date,
    ROUND(w.days_gap, 2) AS days_gap,
    CASE WHEN a.avg_gap > 30 THEN 'At Risk' ELSE 'Active' END AS risk_flag
FROM with_gap w
LEFT JOIN avg_gap a ON a.customer_id = w.customer_id
ORDER BY w.customer_id, w.order_date;
"""
q(sql_q9)

,customer_id,order_date,previous_order_date,days_gap,risk_flag
0,1.0,2024-05-08 11:01:06,None,NaN,At Risk
1,1.0,2024-07-31 00:00:00,2024-05-08 11:01:06,83.54,At Risk
2,1.0,2025-01-20 18:40:39,2024-07-31 00:00:00,173.78,At Risk
3,1.0,2026-05-19 23:26:28,2025-01-20 18:40:39,484.20,At Risk
4,2.0,2024-03-15 19:04:02,None,NaN,At Risk
...,...,...,...,...,...
2832,599.0,2026-02-13 21:37:47,2025-10-04 00:42:50,132.87,At Risk
2833,600.0,2024-03-11 16:59:28,None,NaN,At Risk
2834,600.0,2025-05-27 17:07:45,2024-03-11 16:59:28,442.01,At Risk
2835,600.0,2025-07-13 00:00:00,2025-05-27 17:07:45,46.29,At Risk


In [0]:
sql_q10="""
WITH monthly_revenue AS (
    SELECT
        o.customer_id,
        strftime('%Y-%m', o.order_date) AS month,
        SUM(oi.quantity * oi.unit_price * (1 - oi.discount_percent / 100.0)) AS revenue
    FROM orders o
    JOIN order_items oi ON oi.order_id = o.order_id
    WHERE o.customer_id IS NOT NULL
    GROUP BY o.customer_id, month
),
categorized AS (
    SELECT
        customer_id,
        month,
        revenue,
        CASE
            WHEN revenue > 10000 THEN 'High'
            WHEN revenue BETWEEN 5000 AND 10000 THEN 'Medium'
            ELSE 'Low'
        END AS spend_category
    FROM monthly_revenue
)
SELECT
    month,
    spend_category,
    COUNT(DISTINCT customer_id) AS customer_count
FROM categorized
GROUP BY month, spend_category
ORDER BY month, spend_category;
"""
q(sql_q10)

,month,spend_category,customer_count
0,2024-01,High,59
1,2024-01,Low,10
2,2024-01,Medium,10
3,2024-02,High,47
4,2024-02,Low,12
...,...,...,...
85,2026-05,Low,11
86,2026-05,Medium,6
87,2026-06,High,60
88,2026-06,Low,8


In [0]:
sql_q11="""
WITH lifetime_value AS (
    SELECT
        o.customer_id,
        SUM(oi.quantity * oi.unit_price * (1 - oi.discount_percent / 100.0)) AS total_value
    FROM orders o
    JOIN order_items oi ON oi.order_id = o.order_id
    WHERE o.customer_id IS NOT NULL
    GROUP BY o.customer_id
),
quartiled AS (
    SELECT
        customer_id,
        total_value,
        NTILE(4) OVER (ORDER BY total_value DESC) AS quartile
    FROM lifetime_value
)
SELECT
    customer_id,
    ROUND(total_value, 2) AS total_value,
    quartile,
    CASE quartile
        WHEN 1 THEN 'Platinum'
        WHEN 2 THEN 'Gold'
        WHEN 3 THEN 'Silver'
        WHEN 4 THEN 'Bronze'
    END AS quartile_label
FROM quartiled
ORDER BY quartile, total_value DESC;
"""
q(sql_q11)

,customer_id,total_value,quartile,quartile_label
0,100.0,332702.26,1,Platinum
1,443.0,317403.61,1,Platinum
2,97.0,315980.81,1,Platinum
3,424.0,298273.06,1,Platinum
4,528.0,277768.75,1,Platinum
...,...,...,...,...
582,148.0,1792.26,4,Bronze
583,163.0,-1043.70,4,Bronze
584,355.0,-1350.88,4,Bronze
585,444.0,-2134.22,4,Bronze


In [0]:
sql_q12="""
WITH monthly AS (
    SELECT
        CAST(strftime('%Y', o.order_date) AS INTEGER) AS year,
        CAST(strftime('%m', o.order_date) AS INTEGER) AS month,
        SUM(oi.quantity * oi.unit_price * (1 - oi.discount_percent / 100.0)) AS revenue
    FROM orders o
    JOIN order_items oi ON oi.order_id = o.order_id
    GROUP BY year, month
)
SELECT
    m.year,
    m.month,
    ROUND(m.revenue, 2) AS revenue,
    ROUND(prev.revenue, 2) AS prev_year_revenue,
    CASE
        WHEN prev.revenue IS NULL OR prev.revenue = 0 THEN NULL
        ELSE ROUND((m.revenue - prev.revenue) * 100.0 / prev.revenue, 2)
    END AS yoy_growth_percent
FROM monthly m
LEFT JOIN monthly prev
    ON prev.year = m.year - 1 AND prev.month = m.month
ORDER BY m.year, m.month;
"""
q(sql_q12)

,year,month,revenue,prev_year_revenue,yoy_growth_percent
0,2024,1,1813605.55,NaN,NaN
1,2024,2,1680567.96,NaN,NaN
2,2024,3,2167047.63,NaN,NaN
3,2024,4,1908590.83,NaN,NaN
4,2024,5,2456960.89,NaN,NaN
5,2024,6,1917866.62,NaN,NaN
6,2024,7,2142507.57,NaN,NaN
7,2024,8,2566020.84,NaN,NaN
8,2024,9,2096241.12,NaN,NaN
9,2024,10,2106772.53,NaN,NaN


In [0]:
sql_q13="""
WITH cust_category_orders AS (
    SELECT
        o.customer_id,
        p.category,
        o.order_date,
        FIRST_VALUE(p.category) OVER (
            PARTITION BY o.customer_id ORDER BY o.order_date
            ROWS BETWEEN UNBOUNDED PRECEDING AND UNBOUNDED FOLLOWING
        ) AS first_category,
        LAST_VALUE(p.category) OVER (
            PARTITION BY o.customer_id ORDER BY o.order_date
            ROWS BETWEEN UNBOUNDED PRECEDING AND UNBOUNDED FOLLOWING
        ) AS last_category
    FROM orders o
    JOIN order_items oi ON oi.order_id = o.order_id
    JOIN products p ON p.product_id = oi.product_id
    WHERE o.customer_id IS NOT NULL
)
SELECT DISTINCT
    customer_id,
    first_category,
    last_category AS most_recent_category,
    CASE WHEN first_category != last_category THEN 'Yes' ELSE 'No' END AS category_shift
FROM cust_category_orders
ORDER BY customer_id;
"""
q(sql_q13)

,customer_id,first_category,most_recent_category,category_shift
0,1.0,Clothing,Clothing,No
1,2.0,Books,Home,Yes
2,3.0,Electronics,Clothing,Yes
3,4.0,Electronics,Books,Yes
4,5.0,Clothing,Home,Yes
...,...,...,...,...
582,596.0,Home,Electronics,Yes
583,597.0,Clothing,Clothing,No
584,598.0,Clothing,Clothing,No
585,599.0,Electronics,Home,Yes


In [0]:
sql_q14="""
WITH customer_revenue AS (
    SELECT
        o.customer_id,
        SUM(oi.quantity * oi.unit_price * (1 - oi.discount_percent / 100.0)) AS revenue
    FROM orders o
    JOIN order_items oi ON oi.order_id = o.order_id
    WHERE o.customer_id IS NOT NULL
    GROUP BY o.customer_id
),
totals AS (
    SELECT SUM(revenue) AS grand_total FROM customer_revenue
)
SELECT
    customer_id,
    ROUND(revenue, 2) AS revenue,
    ROUND(SUM(revenue) OVER (ORDER BY revenue DESC ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW), 2) AS cumulative_revenue,
    ROUND(
        SUM(revenue) OVER (ORDER BY revenue DESC ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW)
        * 100.0 / (SELECT grand_total FROM totals), 2
    ) AS cumulative_percent
FROM customer_revenue
ORDER BY revenue DESC;
"""
q(sql_q14)

,customer_id,revenue,cumulative_revenue,cumulative_percent
0,100.0,332702.26,332702.26,0.58
1,443.0,317403.61,650105.87,1.14
2,97.0,315980.81,966086.68,1.70
3,424.0,298273.06,1264359.74,2.22
4,528.0,277768.75,1542128.49,2.71
...,...,...,...,...
582,148.0,1792.26,56908023.01,100.03
583,163.0,-1043.70,56906979.31,100.03
584,355.0,-1350.88,56905628.43,100.03
585,444.0,-2134.22,56903494.22,100.03


In [0]:
sql_q15="""
WITH cohorts AS (
    SELECT
        customer_id,
        strftime('%Y-%m', registration_date) AS cohort_month,
        registration_date
    FROM customers
),
order_months AS (
    SELECT
        o.customer_id,
        o.order_date,
        c.cohort_month,
        c.registration_date,
        CAST(
            (strftime('%Y', o.order_date) - strftime('%Y', c.registration_date)) * 12
            + (strftime('%m', o.order_date) - strftime('%m', c.registration_date))
            AS INTEGER
        ) AS month_offset
    FROM orders o
    JOIN cohorts c ON c.customer_id = o.customer_id
    WHERE o.customer_id IS NOT NULL
),
cohort_sizes AS (
    SELECT cohort_month, COUNT(*) AS cohort_size
    FROM cohorts
    GROUP BY cohort_month
),
cohort_activity AS (
    SELECT
        cohort_month,
        month_offset,
        COUNT(DISTINCT customer_id) AS active_customers
    FROM order_months
    WHERE month_offset BETWEEN 0 AND 3
    GROUP BY cohort_month, month_offset
)
SELECT
    ca.cohort_month,
    ca.month_offset,
    ca.active_customers,
    cs.cohort_size,
    ROUND(ca.active_customers * 100.0 / cs.cohort_size, 2) AS retention_rate_percent
FROM cohort_activity ca
JOIN cohort_sizes cs ON cs.cohort_month = ca.cohort_month
ORDER BY ca.cohort_month, ca.month_offset;
"""
q(sql_q15)

,cohort_month,month_offset,active_customers,cohort_size,retention_rate_percent
0,2023-10,3,1,6,16.67
1,2023-11,2,1,18,5.56
2,2023-11,3,4,18,22.22
3,2023-12,1,3,11,27.27
4,2023-12,2,1,11,9.09
...,...,...,...,...,...
106,2026-04,0,3,15,20.00
107,2026-04,1,2,15,13.33
108,2026-04,2,1,15,6.67
109,2026-05,0,1,19,5.26


In [0]:
sql_q16="""
SELECT
    a.product_id AS product_a,
    b.product_id AS product_b,
    COUNT(*) AS times_bought_together
FROM order_items a
JOIN order_items b
    ON a.order_id = b.order_id
    AND a.product_id < b.product_id
GROUP BY a.product_id, b.product_id
HAVING times_bought_together > 1
ORDER BY times_bought_together DESC
LIMIT 20;
"""
q(sql_q16)

,product_a,product_b,times_bought_together
0,34,143,6
1,37,77,6
2,14,27,5
3,17,49,5
4,21,59,5
5,41,64,5
6,49,134,5
7,50,116,5
8,60,67,5
9,62,78,5


In [0]:
import shutil

shutil.copy("/tmp/ecommerce.db",
            "/Volumes/assignment8/assignment8schema/assignment8volume/ecommerce.db")
conn.close()